In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("=== 🚀 فاز ۳: مقایسه جامع مدل‌های غیرخطی و درختی ===")

# ۱. بارگذاری دیتاست مرجع (دیتاست مهندسی‌شده فازهای قبل)
# مطمئن شوید نام فایل با فایل ذخیره شده شما یکی است
df_master = pd.read_csv('../data/processed/NHANES_Master_Dataset.csv')

# تبدیل Gender به متغیر عددی در صورت نیاز
if 'Gender' in df_master.columns:
    df_master['Is_Male'] = df_master['Gender'].map({'Male': 1, 'Female': 0})

# ۲. تعیین ویژگی‌ها و هدف
features = [
    'Age', 'Is_Male', 'Weight_kg', 'Height_cm', 'BMI', 
    'Body_Fat_Percent', 'Lean_Mass_kg', 'Activity_Score', 
    'Daily_Protein_Intake_g', 'Genetic_Score'
]

X = df_master[features]
y = df_master['Protein_Requirement_g']

# ۳. تقسیم داده به آموزش و تست (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ۴. تعریف مدل‌ها
models = {
    'Linear Regression (Baseline)': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1),
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, verbose=0, random_state=42)
}

# ۵. حلقه‌ی آموزش و ارزیابی
results = []

for name, model in models.items():
    print(f"🔄 در حال آموزش مدل: {name} ...")
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    
    # محاسبه معیارها
    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, predictions)
    
    results.append({
        'Model': name,
        'R² Score': round(r2, 4),
        'MAE (g)': round(mae, 4),
        'RMSE (g)': round(rmse, 4)
    })

# ۶. تبدیل نتایج به دیتافریم و نمایش جدول مقایسه
df_results = pd.DataFrame(results).sort_values(by='R² Score', ascending=False)
print("\n📊 جدول مقایسه نهایی عملکرد مدل‌ها:")
print(df_results.to_string(index=False))

# ۷. رسم و ذخیره نمودار مقایسه مدل‌ها برای گیت‌هاب
os.makedirs('reports/figures', exist_ok=True)
plt.figure(figsize=(10, 5))
sns.barplot(data=df_results, x='R² Score', y='Model', palette='coolwarm')
plt.title('Model Comparison - R² Score Performance', fontweight='bold', pad=15)
plt.xlim(0.9, 1.0) # زوم کردن روی بخش بالای دقت برای دیدن تفاوت‌ها
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('reports/figures/04_model_comparison.png', dpi=300)
plt.close()

print("\n[✓] نمودار مقایسه مدل‌ها در مسیر reports/figures/04_model_comparison.png ذخیره شد.")

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("=== 🧪 فاز ۳.۲: پیاده‌سازی مدل‌های کلاسیک Non-Linear Regression ===")

# ۱. بارگذاری داده‌ها
df_master = pd.read_csv('../data/processed/NHANES_Master_Dataset.csv')
if 'Gender' in df_master.columns:
    df_master['Is_Male'] = df_master['Gender'].map({'Male': 1, 'Female': 0})

features = [
    'Age', 'Is_Male', 'Weight_kg', 'Height_cm', 'BMI', 
    'Body_Fat_Percent', 'Lean_Mass_kg', 'Activity_Score', 
    'Daily_Protein_Intake_g', 'Protein_Requirement_g'
]

X = df_master[features]
y = df_master['Protein_Requirement_g']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ۲. تعریف مدل‌های غیرخطی منحصربه‌فرد
# مدل ۱: رگرسیون چندجمله‌ای درجه ۲ (تولید روابط متقاطع X1 * X2 و توان ۲ ویژگی‌ها)
# مدل ۲: Support Vector Regression (SVR) با هسته غیرخطی RBF (نیازمند استانداردسازی داده)
# مدل ۳: K-Nearest Neighbors Regressor (تخمین بر اساس شباهت فیزیولوژیکی افراد)
nonlinear_models = {
    'Polynomial Regression (Degree 2)': make_pipeline(PolynomialFeatures(degree=2), LinearRegression()),
    'Support Vector Regression (RBF SVR)': make_pipeline(StandardScaler(), SVR(C=100, epsilon=0.1)),
    'K-Nearest Neighbors (KNN)': make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5))
}

# ۳. آموزش و ارزیابی
nonlinear_results = []

for name, model in nonlinear_models.items():
    print(f"🔄 در حال پردازش مدل غیرخطی: {name} ...")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    
    nonlinear_results.append({
        'Model': name,
        'R² Score': round(r2, 4),
        'MAE (g)': round(mae, 4),
        'RMSE (g)': round(rmse, 4)
    })

# ۴. خروجی نتایج غیرخطی جدید
df_nonlinear = pd.DataFrame(nonlinear_results)
print("\n📊 نتایج مدل‌های غیرخطی کلاسیک:")
print(df_nonlinear.to_string(index=False))

# ۵. ترکیب با نتایج درختی و ذخیره کل پلات مقایسه‌ای در یک قاب جامع‌تر
# (این کار جدول README شما را برای فازهای بعدی کامل‌تر می‌کند)